# Modelo Predictivo Precio Airbnb | Proyecto Final Ciencia de Datos 
---

## Objetive: 
To analyze Airbnb listings in order to predict listing prices and understand what factors have the biggest impact on said price. 

## Initial strategic decisions
1. *Raw price versus Log price*: The model will predict log prices instead of the raw prices on the listings in order to prioritize treating proportional differences in pricing the same across the board. This decision was made in order to gain a more symmetric, normal distribution so that the linear regression model works better. The tradeoff being the retransformation bias that occurs when converting from the log price back into dollar amounts.
2. *Models*: For this analysis there will only be one general model that predicts price listings regardless of room types. 
3. *Feature engineering*: In order to gain better insight into the factors affecting Airbnb prices there will be extensive feature engineering and the project will have a big focus on this area of the analysis. 

# 1.1 Exploratory Data Analysis
---
To begin the exploratory data analysis we will be answering the following questions:
- How many rows and columns does the dataset have?
- What does each column represent?

In [1]:
# import libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import style
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns 
import csv
import datetime as dt
from utils import utilities

# import data 
excel = pd.read_excel('datosViviendas.xlsx', header = None)

# drop empty cells and convert lines to strings to read as a csv 
lines = excel[0].dropna().astype(str).tolist()
rows = list(csv.reader(lines))
header = rows[0]

# drop rows with incomplete or exceeding fields 
completeRows = [row for row in rows[1:] if len(row) == len(header)]

# Initial EDA 
df = pd.DataFrame(completeRows, columns = header)
variables = list(df.columns)

print(f"The dataset contains {df.shape[0]} rows and {df.shape[1]} columns.\n")
print(f"The dataset contains the following variables: {variables}.\n")
print(f"The first rows of the dataset are:\n{df.head().to_string()}\n")


The dataset contains 63943 rows and 29 columns.

The dataset contains the following variables: ['id', 'log_price', 'property_type', 'room_type', 'amenities', 'accommodates', 'bathrooms', 'bed_type', 'cancellation_policy', 'cleaning_fee', 'city', 'description', 'first_review', 'host_has_profile_pic', 'host_identity_verified', 'host_response_rate', 'host_since', 'instant_bookable', 'last_review', 'latitude', 'longitude', 'name', 'neighbourhood', 'number_of_reviews', 'review_scores_rating', 'thumbnail_url', 'zipcode', 'bedrooms', 'beds'].

The first rows of the dataset are:
         id           log_price property_type        room_type                                                                                                                                                                                                                                                                                                                amenities accommodates bathrooms  bed_type cancellation_

# 1.1 Structural Overview
---
With my beginning analysis I made the following observations about the dataset:

| Variables              | What is it?                                                             | Data Type   |
| ---------------------- | ----------------------------------------------------------------------- | ----------- |
| id                     | Number that identifies the listing.                                     | str         |
| log_price              | Logarithmic price of the listing.                                       | float       |
| property_type          | Type of property (e.g. House/Apartment).                                | categorical |
| room_type              | Type of room (e.g. )                                                    | categorical |
| amenities              | List of amenities (e.g. TV, Wireless Internet, Air Conditioning, etc.)  | FE          |
| accommodates           | Number of people the property accommodates                              | int         |
| bathrooms              | number of bathrooms                                                     | float       |
| bed_type               | Type of bed                                                             | categorical |
| cancellation_policy    | How strict the cancellation policy is (e.g. strict/ moderate/flexible). | categorical |
| cleaning_fee           | Whether or not a cleaning fee exists.                                   | boolean     |
| city                   | The city the property is located in.                                    | categorical |
| description            | A description of the property.                                          | FE          |
| first_review           | Date of the first review.                                               | FE          |
| host_has_profile_pic   | Whether or not the listing host has a pfp.                              | boolean     |
| host_identity_verified | Whether or not the hosts identity is verified.                          | boolean     |
| host_response_rate     | Percent of times host responds to messages.                             | float       |
| host_since             | Date the Airbnb user became host.                                       | FE          |
| instant_bookable       | Whether or not the property is instantly bookable.                      | boolean     |
| last_review            | The last review's date.                                                 | FE          |
| latitude               | Latitude value.                                                         | FE          |
| longitude              | Longitude value.                                                        | FE          |
| name                   | Property/listing name.                                                  | FE          |
| neighbourhood          | What neighbourhood the property is in.                                  | categorical |
| number_of_reviews      | The number of reviews the listing has.                                  | int         |
| review_scores_rating   | The average rating of the review scores.                                | float       |
| thumbnail_url          | Url for posts thumbnail.                                                | FE          |
| zipcode                | Property's zip code.                                                    | FE          |
| bedrooms               | Number of bedrooms the property has.                                    | int         |
| beds                   | Number of beds the property has.                                        | int         |

In [2]:
# Plan out what dtypes each column should be converted to for analysis and modeling
numeric_columns = ['accommodates', 'number_of_reviews', 'bedrooms', 'beds']
float_columns = ['log_price', 'bathrooms', 'review_scores_rating', 'host_response_rate']
categorical_columns = ['property_type', 'room_type', 'bed_type', 'cancellation_policy', 'city', 'neighbourhood']
boolean_columns = ['cleaning_fee', 'host_has_profile_pic', 'host_identity_verified', 'instant_bookable', ]
feature_engineering_columns = ['amenities', 'description', 'first_review', 'host_since', 'last_review', 'latitude', 'longitude', 'name', 'thumbnail_url', 'zipcode']

# Visualize unique values to determine if any erroneous values exist 
for col in numeric_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

for col in float_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

for col in categorical_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

for col in boolean_columns:
    print(f"Unique values for {col}:\n{df[col].unique()}\n")

Unique values for accommodates:
<StringArray>
[ '3',  '7',  '5',  '4',  '2',  '6',  '8',  '1',  '9', '10', '16', '12', '11',
 '14', '13', '15']
Length: 16, dtype: str

Unique values for number_of_reviews:
<StringArray>
[  '2',   '6',  '10',   '0',   '4',   '3',  '15',   '9', '159',  '82',
 ...
 '243', '382', '380', '358', '265', '354', '376', '315', '1.0', '341']
Length: 356, dtype: str

Unique values for bedrooms:
<StringArray>
[ '1.0',  '3.0',  '2.0',  '0.0',  '4.0',     '',  '5.0',  '6.0',  '7.0',
  '8.0',  '9.0', '10.0',  'NYC']
Length: 13, dtype: str

Unique values for beds:
<StringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                              

Exploring each variable individually I observed some erroneous data that was in incorrect columns or in a bad format and will need to be dealt with.

| Variable               | Erroneous Data                                                                                                                                                                      |
| ---------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| property_type          | Not erroneous but strange categories like tipi, castle, earth house, cave, train, island, yurt, timeshare, lighthouse (maybe yurt is erroneous does not seem like a property type). |
| bathrooms              | Empty strings.                                                                                                                                                                      |
| host_has_profile_pic   | Empty strings, 'Wireless Internet""'                                                                                                                                                |
| host_identity_verified | Empty strings, 'Air conditioning""'                                                                                                                                                 |
| host_response_rate     | 'Kitchen', empty strings.                                                                                                                                                           |
| instant_bookable       | 'Family/kid friendly""'                                                                                                                                                             |
| neighbourhood          | so many different values its very hard to actually tell if any are erroneous like hell's kitchen.                                                                                   |
| number_of_reviews      | 1.0 (all other observed values were ints and it is illogical to have decimal reviews).                                                                                              |
| review_scores_rating   | Empty strings, 'Real Bed'.                                                                                                                                                          |
| bedrooms               | Empty strings, NYC.                                                                                                                                                                 |
| beds                   | Empty strings, a listing description.                                                                                                                                               |


In [3]:
# Find index of possible misaligned row
misaligned_row_index = df[df['host_has_profile_pic'] == 'Wireless Internet""'].index
print(f"The misaligned row is:\n{df.iloc[misaligned_row_index].to_string()}\n")

df = df.drop(misaligned_row_index)

# See number of boolean values that are empty strings

utilities.check_value_counts(df, boolean_columns)
utilities.check_value_counts(df, numeric_columns)
utilities.check_value_counts(df, float_columns)
utilities.check_value_counts(df, categorical_columns)

The misaligned row is:
            id           log_price property_type     room_type                                                                                                                                                                                                                                                                       amenities accommodates bathrooms  bed_type cancellation_policy cleaning_fee city                                                                                                                                                                                                                                                                                                                                                                                                                                        description first_review host_has_profile_pic host_identity_verified host_response_rate host_since       instant_bookable       last_review         

# Observations on individual variables
---

**Misaligned row**: After confirming the initial csv parser filtered out most misaligned rows and only a single one was still present, I decided to drop it since it has a negligible impact on the 63,943 rows. Because of this, the returns were deemed insufficient in relation to the resources needed to fix the parsing error.

**Empty strings**: Observing the 159 empty strings in boolean data types, they were decided to be kept as missing values to follow up later on in the analysis to determine if the values are MAR, MCAR, MNAR. As for the empty strings in other variables they were intentionally preserved and flagged as NaN to analyze further.

**property_type threshold**: Will explore establishing a threshold so categories with few listings go to an 'others' category while takinng into consideration rare types of properties could be an indicator as to the listing's price.


In [4]:
# Map boolean values to True or False and handle empty strings 
boolean_mapping = {'t': True, 'f': False, 'True': True, 'False': False, '': np.nan}
empty_string_handling = {'': np.nan}
chars_to_replace = {'%': ''}

# Transform columns to their corresponding dtypes
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype('Int64')

for col in categorical_columns:
    df[col] = df[col].replace(empty_string_handling).astype('category')

for col in boolean_columns:
    df[col] = df[col].replace(boolean_mapping).astype('boolean')

df['log_price'] = pd.to_numeric(df['log_price'].replace(empty_string_handling), errors="coerce").astype('float64')
df['bathrooms'] = pd.to_numeric(df['bathrooms'].replace(empty_string_handling), errors="coerce").astype('float64')
df['review_scores_rating'] = pd.to_numeric(df['review_scores_rating'].replace(empty_string_handling), errors="coerce").astype('float64')
df['host_response_rate'] = pd.to_numeric(df['host_response_rate'].str.replace(chars_to_replace, regex=False), errors="coerce").astype('float64') / 100

print(f"The dataset contains the following data types after transformation:\n{df.dtypes.to_string()}")

utilities.check_value_counts(df, boolean_columns)
utilities.check_value_counts(df, numeric_columns)
utilities.check_value_counts(df, float_columns)
utilities.check_value_counts(df, categorical_columns)


The dataset contains the following data types after transformation:
id                             str
log_price                  float64
property_type             category
room_type                 category
amenities                      str
accommodates                 Int64
bathrooms                  float64
bed_type                  category
cancellation_policy       category
cleaning_fee               boolean
city                      category
description                    str
first_review                   str
host_has_profile_pic       boolean
host_identity_verified     boolean
host_response_rate         float64
host_since                     str
instant_bookable           boolean
last_review                    str
latitude                       str
longitude                      str
name                           str
neighbourhood             category
number_of_reviews            Int64
review_scores_rating       float64
thumbnail_url                  str
zipcode               